# Real vs synthetic L0 — band-by-band comparison

Compare the **official public L0 product** (downloaded into
`<store>/inputs/public-data/level-0/`) against **our generated canonical L0** (`<store>/l0/`),
band by band: columns 1–2 show the two images, column 3 overlays their DN statistics.

- Real product layout: `measurements/d{DD}/b{bb}/img` (uint16 DN, 12 detectors).
- Our layout: `measurements/d01/b{bb}/band{NN}` (uint16 DN, the packaged datatake).
- The scenes differ (different date/orbit/site), so this compares **distribution character** —
  dynamic range, pedestal, noise floor, texture — not pixel values.
- The two `S02MSIL0P_*` companions carry geometry/quality annotations (footprint lat/lon,
  line-quality masks, detector `msk` grids), **no image DN** — inventoried below, not compared.

In [ ]:
# One-time kernel setup — safe to re-run. After a first-time install, RESTART the kernel.
try:
    import s2_msi_raw_generator  # noqa: F401
    print("s2_msi_raw_generator OK")
except ModuleNotFoundError:
    import sys
    !{sys.executable} -m pip install --quiet -e .. --no-deps matplotlib zarr
    print("installed — now RESTART the kernel (Kernel ▸ Restart Kernel…) and run all cells again")

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from s2_msi_raw_generator import sensor
from s2_msi_raw_generator.io import _open

plt.rcParams["figure.dpi"] = 75

STORE = Path(os.environ.get("S2_DATA_STORE") or "~/data-store").expanduser()
LINES = 1024      # window: first N lines of each product
REAL_DET = 1      # detector of the real product (1–12); ours carries d01 only

real_dir = STORE / "inputs" / "public-data" / "level-0"
reals = sorted(real_dir.glob("S02MSIL0__*.zarr*")) if real_dir.is_dir() else []
REAL = reals[0] if reals else None
syns = sorted(p for p in (STORE / "l0").glob("S02MSIL0__*.zarr") if "_OC" not in p.name) if (STORE / "l0").is_dir() else []
SYN = syns[0] if syns else None
print(f"real:      {REAL.name if REAL else 'NOT FOUND — download into ' + str(real_dir)}")
print(f"synthetic: {SYN.name if SYN else 'NOT FOUND — run: python scripts/run_pipeline.py'}")

_real_g = _open(str(REAL)) if REAL else None
_syn_g = _open(str(SYN)) if SYN else None


def read_real(band, det=REAL_DET, lines=LINES):
    a = _real_g[f"measurements/d{det:02d}/{sensor.zarr_band_key(band)}/img"]
    return np.asarray(a[:lines])


def read_syn(band, lines=LINES):
    bkey = sensor.zarr_band_key(band)
    a = _syn_g[f"measurements/d01/{bkey}/band{sensor.band_number(band)}"]
    return np.asarray(a[:lines])


def stretch(img):
    im = np.asarray(img, dtype=np.float64)
    lo, hi = np.percentile(im, (2, 98))
    return np.clip(im, lo, max(hi, lo + 1e-9))


def dn_stats(a):
    a = np.asarray(a, dtype=np.float64)
    return dict(mean=float(a.mean()), std=float(a.std()),
                p2=float(np.percentile(a, 2)), p98=float(np.percentile(a, 98)),
                mx=float(a.max()))

## Band-by-band: real | synthetic | DN statistics

Each row: the real product's window (left), ours (middle), and the DN histograms overlaid with
μ/σ and the 2–98 % range (right). Both images are independently percentile-stretched.

In [ ]:
rows = []
if REAL and SYN:
    for bn in sensor.BANDS:
        try:
            r, s = read_real(bn), read_syn(bn)
        except KeyError as exc:
            print(f"{bn}: skipped ({exc})")
            continue
        sr, ss = dn_stats(r), dn_stats(s)
        rows.append((bn, sr, ss))
        fig, ax = plt.subplots(1, 3, figsize=(14, 2.9))
        ax[0].imshow(stretch(r), cmap="gray", aspect="auto", interpolation="nearest")
        ax[0].set_title(f"{bn} — real {REAL.name.split('_')[1][:8]} d{REAL_DET:02d} {r.shape}", fontsize=8)
        ax[1].imshow(stretch(s), cmap="gray", aspect="auto", interpolation="nearest")
        ax[1].set_title(f"{bn} — synthetic d01 {s.shape}", fontsize=8)
        for a in ax[:2]:
            a.set_xticks([]); a.set_yticks([])
        ax[2].hist(r.ravel(), bins=120, density=True, alpha=0.55, label="real", color="tab:blue")
        ax[2].hist(s.ravel(), bins=120, density=True, alpha=0.55, label="synthetic", color="tab:orange")
        ax[2].legend(fontsize=7, loc="upper right")
        ax[2].set_title(
            f"real μ{sr['mean']:6.0f} σ{sr['std']:5.0f} [{sr['p2']:.0f}–{sr['p98']:.0f}]   "
            f"syn μ{ss['mean']:6.0f} σ{ss['std']:5.0f} [{ss['p2']:.0f}–{ss['p98']:.0f}]",
            fontsize=7.5,
        )
        ax[2].set_xlabel("DN", fontsize=7); ax[2].tick_params(labelsize=7)
        plt.tight_layout(); plt.show()
else:
    print("need both products — see the setup cell above")

## All-band summary

Mean ± σ per band side by side, and the full statistics table.

In [ ]:
if rows:
    bands = [bn for bn, _, _ in rows]
    x = np.arange(len(bands))
    rm = np.array([sr["mean"] for _, sr, _ in rows]); rs = np.array([sr["std"] for _, sr, _ in rows])
    sm = np.array([ss["mean"] for _, _, ss in rows]); sst = np.array([ss["std"] for _, _, ss in rows])
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.2))
    ax[0].bar(x - 0.2, rm, 0.4, yerr=rs, label="real", color="tab:blue", capsize=2)
    ax[0].bar(x + 0.2, sm, 0.4, yerr=sst, label="synthetic", color="tab:orange", capsize=2)
    ax[0].set_xticks(x, bands, fontsize=7); ax[0].set_ylabel("DN"); ax[0].legend(fontsize=8)
    ax[0].set_title("mean DN ± σ per band", fontsize=9)
    ax[1].plot(x, rs, "o-", label="real σ", color="tab:blue", lw=1)
    ax[1].plot(x, sst, "o-", label="synthetic σ", color="tab:orange", lw=1)
    ax[1].set_xticks(x, bands, fontsize=7); ax[1].set_ylabel("σ [DN]"); ax[1].legend(fontsize=8)
    ax[1].set_title("DN spread per band", fontsize=9)
    plt.tight_layout(); plt.show()

    print(f"{'band':5s} | {'real μ':>7s} {'σ':>6s} {'p2–p98':>13s} {'max':>5s} | "
          f"{'syn μ':>7s} {'σ':>6s} {'p2–p98':>13s} {'max':>5s}")
    print("-" * 88)
    for bn, sr, ss in rows:
        print(f"{bn:5s} | {sr['mean']:7.1f} {sr['std']:6.1f} "
              f"[{sr['p2']:5.0f}–{sr['p98']:5.0f}] {sr['mx']:5.0f} | "
              f"{ss['mean']:7.1f} {ss['std']:6.1f} "
              f"[{ss['p2']:5.0f}–{ss['p98']:5.0f}] {ss['mx']:5.0f}")

## Column profile — fixed-pattern / PRNU signature

The along-track (column) mean exposes the per-detector fixed pattern: the real product carries
the instrument's PRNU + dark structure, ours carries whatever the chain impressed.

In [ ]:
PROFILE_BAND = "B04"
if REAL and SYN:
    r, s = read_real(PROFILE_BAND), read_syn(PROFILE_BAND)
    fig, ax = plt.subplots(2, 1, figsize=(13, 4.6), sharex=False)
    pr, ps = r.mean(axis=0), s.mean(axis=0)
    ax[0].plot(pr, lw=0.5, color="tab:blue", label=f"real d{REAL_DET:02d}")
    ax[0].plot(ps, lw=0.5, color="tab:orange", alpha=0.8, label="synthetic d01")
    ax[0].set_title(f"{PROFILE_BAND} — column mean DN (fixed pattern)", fontsize=9)
    ax[0].legend(fontsize=8); ax[0].tick_params(labelsize=7)
    k = max(3, min(51, (min(pr.size, ps.size) // 2) * 2 - 1))  # odd, never wider than the data
    e = k // 2
    hp_r = pr - np.convolve(pr, np.ones(k) / k, mode="same")
    hp_s = ps - np.convolve(ps, np.ones(k) / k, mode="same")
    ax[1].plot(hp_r, lw=0.5, color="tab:blue", label=f"real (σ={hp_r[e:-e].std():.2f} DN)")
    ax[1].plot(hp_s, lw=0.5, color="tab:orange", alpha=0.8, label=f"synthetic (σ={hp_s[e:-e].std():.2f} DN)")
    ax[1].set_title("high-pass (scene removed) — column FPN amplitude", fontsize=9)
    ax[1].set_xlabel("detector column", fontsize=8); ax[1].legend(fontsize=8); ax[1].tick_params(labelsize=7)
    plt.tight_layout(); plt.show()

## The `S02MSIL0P_*` companions (annotations, no image DN)

In [ ]:
for p in (sorted(real_dir.glob("S02MSIL0P_*.zarr*")) if real_dir.is_dir() else []):
    g = _open(str(p))
    d01 = g["d01"] if "d01" in dict(g.groups()) else g
    parts = []
    for name, sub in sorted(d01.groups(), key=lambda kv: kv[0]):
        parts.append(f"{name}/[{', '.join(n for n, _ in sorted(sub.arrays())) or ', '.join(sorted(dict(sub.groups())))}]")
    for name, arr in sorted(d01.arrays(), key=lambda kv: kv[0]):
        parts.append(f"{name} {arr.shape}")
    print(f"{p.name}\n    d01 → {'; '.join(parts) if parts else '(see product tree)'}")

## Same-scene bit-exact check

If the canonical L0 was produced through `S2_E2ES_PHASES=import-l0,...`, this section compares the canonical ground-decoded DN directly with the public source array (A3). Without import provenance the notebook remains in cross-scene distribution-comparison mode.

In [ ]:
from s2_msi_raw_generator import l0product

same_scene_rows = []
if _syn_g is not None:
    prov = dict(_syn_g.attrs).get("other_metadata", {}).get("import_provenance")
    if prov:
        REAL_DET = int(prov.get("source_detector", REAL_DET))
        source = _open(prov["source_product"])
        for band in prov.get("bands", {}):
            public = np.asarray(source[f"measurements/d{REAL_DET:02d}/{sensor.zarr_band_key(band)}/img"], dtype=np.uint16)
            ours = l0product.read_l0_isp_dn(str(SYN), 1, band)
            same_scene_rows.append((band, bool(np.array_equal(public, ours)), int(np.count_nonzero(public != ours))))

if same_scene_rows:
    print("SAME_SCENE=True")
    print("band  array_equal  diff_px")
    for row in same_scene_rows:
        print(f"{row[0]:>4}  {str(row[1]):>11}  {row[2]}")
else:
    print("SAME_SCENE=False — cross-scene distribution-level comparison only")